# 🌍 Weorold: The Ultimate GPU Planet Generator

This is the central entry point for the **Weorold** project in Google Colab. It leverages NVIDIA GPUs to simulate climate, tectonics, and massive hydraulic erosion.

### 🚀 1. Setup Environment
Verify GPU acceleration and install required libraries.

In [ ]:
# @title 🚀 1. Setup & Clone Repository
repo_url = "https://github.com/pmcray/weorold.git" # @param {type:"string"}

import os
import subprocess
import sys

def setup_environment():
    # 1. Clone Repo if not present
    repo_name = repo_url.split("/")[-1].replace(".git", "")
    if not os.path.exists(repo_name):
        print(f"📂 Cloning {repo_url}...")
        subprocess.run(["git", "clone", repo_url])
    
    if os.path.exists(repo_name):
        os.chdir(repo_name)
        if os.getcwd() not in sys.path:
            sys.path.append(os.getcwd())
        print(f"✅ Working directory set to: {os.getcwd()}")

    # 2. Detect and Install CuPy
    print("🔍 Detecting CUDA version...")
    try:
        cuda_version_raw = subprocess.check_output(["nvcc", "--version"]).decode("utf-8")
        pkg = "cupy-cuda12x" if "12." in cuda_version_raw else "cupy-cuda11x" if "11." in cuda_version_raw else "cupy"
        print(f"📦 Installing {pkg}...")
        subprocess.run(["pip", "install", "-q", pkg, "opencv-python-headless", "scipy"])
    except:
        subprocess.run(["pip", "install", "-q", "cupy", "opencv-python-headless", "scipy"])

    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"])
    print("✨ Environment Ready!")

setup_environment()

import cupy as cp
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video, display, FileLink

try:
    print(f"\n✅ GPU Verified: {cp.cuda.Device(0).attributes['MultiProcessorCount']} Multiprocessors active.")
except Exception as e:
    print(f"❌ GPU Error: {e}. Check Runtime > Change runtime type > T4 GPU.")

### 🛠️ 2. The Planet Generation Pipeline
This pipeline now includes **Rain Shadows**, **Whittaker Biomes**, and **Vegetation Scattering**.

In [ ]:
# @title Customize Your World { run: "auto" }
planet_name = "aethelgard" # @param {type:"string"}
erosion_particles = 1000000 # @param {type:"slider", min:100000, max:5000000, step:100000}
tectonic_epochs = 5 # @param {type:"slider", min:1, max:20, step:1}
temperature_bias = 0.5 # @param {type:"slider", min:0.0, max:1.0, step:0.1}
moisture_bias = 0.6 # @param {type:"slider", min:0.0, max:1.0, step:0.1}
export_for_unity = True # @param {type:"boolean"}

import wp1_fractal_coastline as wp1
import wp2_heightmap_synthesis as wp2
import wp3_biome_texturing as wp3
import wp4_cloud_layer as wp4
import wp5_final_renderer as wp5
import wp9_tectonics as wp9
import wp10_hydrology as wp10
import wp11_erosion_gpu as wp11_gpu
import wp13_vegetation as wp13
import wp14_unity_export as wp14
import weorold_ultra as globe

def run_ultimate_pipeline():
    print(f"--- Generating {planet_name} ---")
    
    # 1. Coastline
    wp1.process_sketch_to_fractal_mask(None, upscale_factor=4)
    
    # 2. Heightmap Synthesis
    wp2.synthesize_heightmap('wp1_fractal_mask.png')
    
    # 3. Tectonic Evolution
    base_h16 = cv2.imread('wp2_height_map.png', cv2.IMREAD_UNCHANGED)
    tectonic_h16 = wp9.generate_tectonic_evolution(base_h16, time_steps=tectonic_epochs)
    
    # 4. GPU Hydraulic Erosion
    print(f"Eroding with {erosion_particles} particles...")
    eroded_h16 = wp11_gpu.simulate_hydraulic_erosion_gpu(tectonic_h16, num_particles=erosion_particles)
    cv2.imwrite(f'{planet_name}_height_map.png', eroded_h16)
    
    # 5. Climate & Biomes (Includes Rain Shadows)
    wp3.create_surface_texture(f'{planet_name}_height_map.png', 
                               temperature=temperature_bias, 
                               moisture=moisture_bias, 
                               output_prefix=f'{planet_name}_wp3')
    
    # 6. Hydrology
    river_map, lake_map = wp10.simulate_hydrology(eroded_h16, mask_path='wp1_fractal_mask.png')
    texture = cv2.imread(f'{planet_name}_wp3_surface_texture.png')
    hydrology_texture = wp10.apply_hydrology_to_texture(cv2.cvtColor(texture, cv2.COLOR_BGR2RGB), river_map, lake_map)
    final_tex_path = f'{planet_name}_final_texture.png'
    cv2.imwrite(final_tex_path, cv2.cvtColor(hydrology_texture, cv2.COLOR_RGB2BGR))
    
    # 7. Vegetation Scatter Maps
    wp13.generate_vegetation_scatter(f'{planet_name}_wp3_biome_map.png', 
                                     f'{planet_name}_wp3_temperature_map.png', 
                                     f'{planet_name}_wp3_moisture_map.png', 
                                     output_prefix=f'{planet_name}_wp13')
    
    # 8. Rendering & Globe
    wp4.create_cloud_layer(eroded_h16.shape)
    wp5.render_final_maps(texture_path=final_tex_path, 
                          heightmap_path=f'{planet_name}_height_map.png', 
                          cloud_path='wp4_cloud_map.png', 
                          mask_path='wp1_fractal_mask.png', 
                          output_prefix=f'{planet_name}_wp5')
    
    video_path = f'{planet_name}_orbit.mp4'
    globe.create_rotating_globe(f'{planet_name}_wp5_shaded_clouds.png', video_path, frames=60)

    # 9. Unity Export
    if export_for_unity:
        zip_path = wp14.export_for_unity(planet_name, 
                                         f'{planet_name}_height_map.png', 
                                         f'{planet_name}_wp3_biome_map.png', 
                                         f'{planet_name}_wp3_moisture_map.png', 
                                         f'{planet_name}_wp3_temperature_map.png')
        print(f"Unity bundle ready: {zip_path}")
        display(FileLink(zip_path))

    # --- MAP INSPECTION ---
    print("\n🖼️ Inspecting generated maps...")
    fig, axes = plt.subplots(2, 2, figsize=(20, 10))
    
    # Coastline Mask
    mask_img = cv2.imread('wp1_fractal_mask.png', cv2.IMREAD_GRAYSCALE)
    if mask_img is not None:
        axes[0, 0].imshow(mask_img, cmap='gray')
        axes[0, 0].set_title("Coastline Mask")
    
    # Heightmap
    h_img = cv2.imread(f'{planet_name}_height_map.png', cv2.IMREAD_UNCHANGED)
    if h_img is not None:
        axes[0, 1].imshow(h_img, cmap='terrain')
        axes[0, 1].set_title("Final Heightmap (Eroded)")
    
    # Biome Map
    biome_img = cv2.imread(f'{planet_name}_wp3_biome_map.png')
    if biome_img is not None:
        axes[1, 0].imshow(cv2.cvtColor(biome_img, cv2.COLOR_BGR2RGB))
        axes[1, 0].set_title("Whittaker Biome Map")
    
    # Final Shaded Map
    shaded_img = cv2.imread(f'{planet_name}_wp5_shaded_clouds.png')
    if shaded_img is not None:
        axes[1, 1].imshow(cv2.cvtColor(shaded_img, cv2.COLOR_BGR2RGB))
        axes[1, 1].set_title("Final Shaded Texture (Globe Source)")
    
    for ax in axes.flatten(): ax.axis('off')
    plt.tight_layout()
    plt.show()

    display(Video(video_path, embed=True, width=600))
    print("\n✨ Planet '{planet_name}' is complete!")

run_ultimate_pipeline()